## composition

This notebook introduces `part` usage (composition); after running it you can declare that a system definition owns named instances of its subsystem types.

The previous notebook established that `HeatingSystem` and `ControlSystem` are specializations of `ToastingSystem`. This notebook composes them into a `Toaster`: a system that owns a heating part and a control part. The model is now structurally complete for Chapter 1.

In [ ]:
import opensysml
from toaster.report import format_diagnostics
from toaster.render import model_to_dot, render_dot

source = """
package ToasterDemo {
    private import ScalarValues::*;

    abstract part def ToastingSystem {
        doc /* Transform bread into toast acceptable to its user. */
    }

    part def Heater {
        attribute power : Real default = 800.0;
    }

    part def HeatingSystem :> ToastingSystem;
    part def ControlSystem :> ToastingSystem;

    part def Toaster {
        attribute cycleTime : Real default = 120.0;
        part heating : HeatingSystem;
        part control : ControlSystem;
    }
}
"""

conn = opensysml.connect(version="v0.9.0")
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

In [ ]:
# Negative control: a part usage must name a type that exists in the model.
# Composing an undefined type raises "unresolved reference".
bad_source = """
package Bad {
    part def Toaster {
        part heating : UndefinedSubsystem;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print("Expected error:", bad.diagnostics[0].message)

In [ ]:
toaster = model.find("ToasterDemo::Toaster")
assert toaster is not None

parts = toaster.parts()
attrs = toaster.attributes()
print(f"Toaster parts ({len(parts)}):")
for p in parts:
    print(f"  {p.id}")
print(f"Toaster attributes ({len(attrs)}):")
for a in attrs:
    print(f"  {a.id}")

# Generate and print the part-hierarchy DOT diagram
dot_src = model_to_dot(model, title="ToasterDemo")
print()
print(dot_src[:400])
conn.close()

`part def Toaster { part heating : HeatingSystem; part control : ControlSystem; }` is the A-F composition; OpenSysML resolves each part usage to its typed definition (O-S); `toaster.parts()` returns the two part symbols (E).

Try the chapter exercise in `exercises/ch01/exercise.ipynb`: compose a `CoffeeMaker` from `BrewUnit` and `HeatExchanger` and verify both parts appear via `parts()`.